# Bronze - VRA (Voo Regular Ativo)
### Lê os 12 CSVs mensais do volume voebem.bronze.arquivos/vra/ e materializa voebem.bronze.vra

Regras da camada Bronze:
- Nada de tipagem - tudo string, exatamente como veio do arquivo;
- Nada de filtro - nenhuma linha é descartada;
- Colunas de auditoria - de cada arquivo veio e quando foi ingerido;
- idempotente - rodar duas vezes não duplica.

In [0]:
from pyspark.sql import functions as F

CAMINHO = "/Volumes/voebem/bronze/arquivos/vra/*.csv"
TABELA = "voebem.bronze.vra"





# Leitura

Quatro opções resolvem problemas de arquivo:

| opção | resolve |
|-------|----------|
| sep=";" | separador brasileiro, não vírgula |
| skipRows=1 | a 1ª linha é Atualizado em: \<data\>, não o cabeçalho — e o BOM tá aa aí também, some junto |
| header=true | a 2ª linha (a primeira que sobra) é o cabeçalho de verdade |
| inferSchema **desligado** (default) | bronze não tipa: tudo chega como string |




In [0]:
bruto = (
    spark.read.format("csv")
    .option("sep", ";")
    .option("header", True)
    .option("skipRows", 1)      # descarta "Atualizado em: ..."
    .option("inferSchema", False)   # bronze não tipa: tudo string
    .option("quote", "\0")          # desabilita quote
    .option("escape", "\0")         # desabilita escape
    .option("encoding", "UTF-8")
    .option("mode", "PERMISSIVE")   # bronze não descarta linha nenhuma
    .load(CAMINHO)
)

print("Colunas lidas do arquivo: ")
for c in bruto.columns:
    print(f"{c!r}")
